# Offside AI — Colab quickstart

Student research prototype. **Run the audio stage first** on 3–4 short clips, then the full pipeline.

Runtime → Change runtime type → **GPU** (optional but recommended for RF-DETR / ViTPose / YOLO).

## 1. Install

If you cloned the repo into Drive / Colab, set `REPO` accordingly. On a fresh Colab, clone from GitHub.

In [ ]:
import os, sys, subprocess
REPO = os.environ.get("OFFSIDE_REPO", "/content/offside-ai-prototype")
if not os.path.isdir(REPO):
    # Replace with your fork URL if needed
    !git clone https://github.com/ka-dm/offside-ai-prototype.git $REPO
%cd {REPO}
!apt-get -qq update && apt-get -qq install -y ffmpeg >/dev/null
!pip -q install -r requirements-colab.txt
!pip -q install -e .
print("cwd=", os.getcwd())

## 2. Upload a short clip

In [ ]:
from google.colab import files
import pathlib
uploaded = files.upload()
assert uploaded, "Upload at least one video"
CLIP = pathlib.Path(list(uploaded.keys())[0]).resolve()
print("Using", CLIP)

## 3. Audio stage (novelty) — inspect spectrogram & pass frame

In [ ]:
from IPython.display import Image, display
from offside_ai.audio.detect import AudioPassConfig, detect_pass_from_audio
from offside_ai.audio.plotting import plot_spectrogram_with_peaks
from offside_ai.common import get_video_meta, read_frame
import cv2
from pathlib import Path

OUT = Path("artifacts/colab_audio")
OUT.mkdir(parents=True, exist_ok=True)
meta = get_video_meta(CLIP)
print(meta)

cfg = AudioPassConfig(av_offset_s=0.0)  # tune if A/V skewed
audio = detect_pass_from_audio(CLIP, fps=meta["fps"], cfg=cfg, work_dir=OUT)
plot_path = OUT / "spectrogram_peaks.png"
plot_spectrogram_with_peaks(
    audio.spectrogram_db, audio.spectrogram_times_s, audio.spectrogram_freqs_hz,
    audio.onset_times_s, onset_env=audio.onset_env, onset_times_s=audio.onset_env_times_s,
    output_path=plot_path,
)
print("best", audio.best)
display(Image(filename=str(plot_path)))
if audio.best:
    frame = read_frame(CLIP, audio.best.frame_index)
    frame_path = OUT / "pass_frame.jpg"
    cv2.imwrite(str(frame_path), frame)
    display(Image(filename=str(frame_path)))
    if audio.best.confidence < 0.45:
        print("LOW CONFIDENCE — set MANUAL_PASS_FRAME below and re-run pipeline.")

## 4. Full pipeline

Start with `mock` backends to validate wiring; switch to `rfdetr`/`yolo` + `vitpose` after weights download.

In [ ]:
from offside_ai.pipeline.run import PipelineConfig, run_pipeline
from IPython.display import Image, display

MANUAL_PASS_FRAME = None  # e.g. 42 if audio is wrong
cfg = PipelineConfig(
    detector_backend="mock",  # try "rfdetr" or "yolo" on GPU
    pose_backend="mock",      # try "vitpose"
    pitch_backend="mock",     # or supply manual_pitch_points
    attacking_team_id=0,
    attacking_direction="positive_x",
    av_offset_s=0.0,
    manual_pass_frame=MANUAL_PASS_FRAME,
    work_dir="artifacts/colab_pipeline",
)
result = run_pipeline(CLIP, cfg)
print(result.decision.verdict)
print(result.pass_event)
display(Image(filename=result.spectrogram_path))
display(Image(filename=result.annotated_path))

## 5. Optional Gradio UI

In [ ]:
# from offside_ai.app.gradio_app import build_app
# build_app().launch(share=True)